# 09 · Baseline Sales Forecast

**Goal:** add a transparent forecast benchmark without changing the existing sales-trend analysis.

This notebook aggregates the same Gold sales data by month and uses a simple 3-month moving-average baseline. If the latest calendar month is partial, it is excluded from the forecast calculation.

This is a **baseline forecast**, not a production forecasting model. The assumption is deliberately simple and easy to reproduce.


## 1. Setup


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
import config

os.makedirs("output", exist_ok=True)
engine = create_engine(config.get_conn_url())


## 2. Load and aggregate monthly sales


In [ ]:
sales = pd.read_sql(
    """
    SELECT order_date, sales_amount
    FROM gold.fact_sales
    WHERE order_date IS NOT NULL
    """,
    engine,
)

sales["order_date"] = pd.to_datetime(sales["order_date"])
monthly = (
    sales.assign(order_month=sales["order_date"].dt.to_period("M"))
    .groupby("order_month")["sales_amount"]
    .sum()
    .sort_index()
)

data_end_date = sales["order_date"].max().normalize()
latest_month = monthly.index.max()
latest_month_end = latest_month.to_timestamp(how="end").normalize()
latest_month_is_partial = data_end_date < latest_month_end

if latest_month_is_partial:
    complete_monthly = monthly.iloc[:-1]
    excluded_partial_month = str(latest_month)
else:
    complete_monthly = monthly.copy()
    excluded_partial_month = None

print(f"Data end date: {data_end_date.date()}")
print(f"Excluded partial month: {excluded_partial_month}")


## 3. Three-month moving-average baseline


In [ ]:
baseline_window = complete_monthly.tail(3)
baseline_sales = baseline_window.mean()

forecast_start = latest_month + 1

future_periods = pd.period_range(
    start=forecast_start,
    periods=3,
    freq="M"
)

forecast = pd.DataFrame({
    "forecast_month": future_periods.astype(str),
    "forecast_sales": baseline_sales,
})

print("Baseline window:")
print(baseline_window)
print(f"\nForecast baseline: {baseline_sales:,.0f} per month")
print(forecast)


The dataset ends on 28 January 2014, so January 2014 is treated as a partial month and excluded. The baseline therefore uses October, November and December 2013.


## 4. Visualise the baseline forecast


In [ ]:
history = complete_monthly.tail(18)

plt.figure(figsize=(12, 5))
plt.plot(history.index.astype(str), history.values, marker="o", label="Historical sales")
plt.plot(forecast["forecast_month"], forecast["forecast_sales"], marker="o", linestyle="--", label="3M baseline forecast")
plt.xticks(rotation=45)
plt.ylabel("Sales")
plt.xlabel("Month")
plt.title("Monthly Sales — Baseline Forecast")
plt.legend()
plt.tight_layout()
plt.savefig("output/15_sales_baseline_forecast.png", dpi=160)
plt.show()


## 5. Save outputs


In [ ]:
forecast.to_csv("output/sales_baseline_forecast.csv", index=False)
pd.DataFrame({
    "metric": ["forecast_method", "baseline_months", "excluded_partial_month"],
    "value": ["3-month moving-average baseline", ", ".join(baseline_window.index.astype(str)), excluded_partial_month or "None"],
}).to_csv("output/sales_forecast_method_notes.csv", index=False)
print("Forecast outputs saved.")


## 6. How to use this result

Use the forecast as a benchmark for planning and for comparing future forecasting methods. A later version could test seasonal models or time-series cross-validation, but this portfolio version avoids overclaiming predictive accuracy.
